# 🧹 Sales Data — Normalización + Bot RAG con Mistral Small 4
### Google Colab — Ejecutar las celdas en orden

**Flujo completo:**
- **Celdas 1–12** → Limpieza y normalización profesional del CSV
- **Celdas 13–18** → Motor RAG + Bot interactivo con Mistral Small 4

---

# PARTE 1 — NORMALIZACIÓN DE DATOS
---

## 📦 Celda 1 — Instalación de dependencias

In [3]:
# Normalización: thefuzz para fuzzy matching, openpyxl para Excel
# Bot: mistralai para la API, scikit-learn para el motor TF-IDF
!pip install langchain langchain-experimental langchain-mistralai pandas openpyxl thefuzz python-Levenshtein --quiet
print('✅ Dependencias instaladas')

✅ Dependencias instaladas


## 📚 Celda 2 — Importaciones

In [10]:
import pandas as pd
import numpy as np
import re
import unicodedata
from thefuzz import fuzz, process
from google.colab import files, userdata
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_mistralai import ChatMistralAI
# Nueva ruta de importación para el callback handler:
from langchain_core.callbacks import StdOutCallbackHandler
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas')

✅ Librerías cargadas


## 🛠 Celda 3 — Funciones de normalización

In [11]:
# ── Texto general ─────────────────────────────────────────────

def quitar_tildes(texto):
    texto = unicodedata.normalize('NFKD', str(texto))
    return texto.encode('ascii', 'ignore').decode('ascii')

def limpiar_texto(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def normalizar_texto_lower(texto):
    if pd.isna(texto):
        return np.nan
    texto = quitar_tildes(str(texto))
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def normalizar_nombre_columna(nombre):
    nombre = quitar_tildes(str(nombre))
    nombre = nombre.lower()
    nombre = re.sub(r'[\s\-]+', '_', nombre)
    nombre = re.sub(r'[^\w]', '', nombre)
    return nombre

def normalizar_fecha(fecha):
    if pd.isna(fecha):
        return np.nan
    try:
        return pd.to_datetime(fecha, format='%m/%d/%Y %H:%M', errors='coerce')
    except:
        return pd.to_datetime(fecha, errors='coerce')

def normalizar_telefono(telefono):
    if pd.isna(telefono):
        return np.nan
    telefono = re.sub(r'[^+\d]', '', str(telefono))
    return telefono if telefono else np.nan

print('✅ Funciones de normalización cargadas')

✅ Funciones de normalización cargadas


## 📂 Celda 4 — Carga del archivo CSV

In [12]:
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

# Cargar con encoding correcto
try:
    df = pd.read_csv(nombre_archivo, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(nombre_archivo, encoding='latin-1')
    print('⚠ Archivo leído con latin-1')

print(f'✅ Archivo cargado: {df.shape[0]} filas, {df.shape[1]} columnas')

# Guardar estado original
filas_originales = df.shape[0]
columnas_originales = df.columns.tolist()

Saving sales_data_sample.csv to sales_data_sample.csv
⚠ Archivo leído con latin-1
✅ Archivo cargado: 2823 filas, 25 columnas


## 🔍 CELDA 5 - Normalizar columnas

In [13]:
df.columns = [normalizar_nombre_columna(col) for col in df.columns]
print('✅ Columnas normalizadas a snake_case')

✅ Columnas normalizadas a snake_case


## CELDA 6 - Limpieza general



In [14]:
# Eliminar filas completamente vacías
df.dropna(how='all', inplace=True)

# Eliminar duplicados exactos
df.drop_duplicates(inplace=True)

# Convertir fechas
if 'orderdate' in df.columns:
    df['orderdate'] = df['orderdate'].apply(normalizar_fecha)
    df['order_year_month'] = df['orderdate'].dt.strftime('%Y-%m')
    df['year_id'] = df['orderdate'].dt.year

# Normalizar teléfonos
if 'phone' in df.columns:
    df['phone'] = df['phone'].apply(normalizar_telefono)

# Limpiar columnas de nombre propio
columnas_nombre = ['customername', 'contactlastname', 'contactfirstname',
                   'addressline1', 'city', 'state', 'country']
for col in columnas_nombre:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_texto)

# Normalizar columnas categóricas
columnas_categoricas = ['productline', 'dealsize', 'status', 'territory']
for col in columnas_categoricas:
    if col in df.columns:
        df[col] = df[col].apply(normalizar_texto_lower)

print('✅ Limpieza general completada')

✅ Limpieza general completada


## Celda 7 — Manejo de nulos

In [15]:
# Eliminar columnas con más de 50% nulos
for col in df.columns:
    if df[col].isnull().mean() > 0.5:
        df.drop(columns=[col], inplace=True)
        print(f'🗑 Columna eliminada: {col}')

# Imputar nulos en columnas clave
for col in ['state', 'postalcode', 'territory']:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

# Eliminar filas sin datos críticos
df.dropna(subset=['customername', 'country', 'sales'], inplace=True)

print(f'✅ Dataset después de limpieza: {df.shape[0]} filas, {df.shape[1]} columnas')

🗑 Columna eliminada: addressline2
🗑 Columna eliminada: state
✅ Dataset después de limpieza: 2823 filas, 24 columnas


## Celda 8 — Fuzzy matching (unificación de clientes)

In [16]:
UMBRAL_SIMILITUD = 85

def deduplicar_columna(serie, umbral):
    valores_unicos = serie.dropna().unique().tolist()
    mapa = {}
    procesados = set()

    for valor in valores_unicos:
        if valor in procesados:
            continue

        # Buscar similares
        similares = process.extract(
            valor, valores_unicos,
            scorer=fuzz.token_sort_ratio,
            limit=50  # Limitado para performance
        )

        grupo = [v for v, score in similares if score >= umbral]
        frecuencias = serie[serie.isin(grupo)].value_counts()
        canonico = frecuencias.idxmax()

        for v in grupo:
            mapa[v] = canonico
            procesados.add(v)

    return serie.map(lambda x: mapa.get(x, x))

if 'customername' in df.columns:
    antes = df['customername'].nunique()
    df['customername'] = deduplicar_columna(df['customername'], UMBRAL_SIMILITUD)
    despues = df['customername'].nunique()
    print(f'✅ Clientes unificados: {antes} → {despues} ({antes - despues} fusiones)')

✅ Clientes unificados: 92 → 91 (1 fusiones)


## 🔍 Celda 9 — Feature engineering

In [17]:
# Flag para deals grandes
if 'dealsize' in df.columns:
    df['is_large_deal'] = (df['dealsize'] == 'large').astype(int)

# Margen de ganancia
if all(col in df.columns for col in ['sales', 'quantityordered', 'msrp']):
    df['profit_margin'] = np.where(
        df['sales'] > 0,
        (df['sales'] - df['quantityordered'] * df['msrp']) / df['sales'],
        0
    ).round(4)

print('✅ Feature engineering completado')

✅ Feature engineering completado


# CELDA 10 - Exportar dataset limpio (opcional)

In [ ]:
nombre_salida = 'sales_data_clean.csv'
df.to_csv(nombre_salida, index=False, encoding='utf-8-sig')
files.download(nombre_salida)
print(f'✅ Dataset limpio exportado: {nombre_salida}')

# CELDA 11 - Configurar API Key

In [18]:
try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    if not MISTRAL_API_KEY:
        raise ValueError("Secret vacío")
    print('✅ API key cargada desde Colab Secrets')
except:
    MISTRAL_API_KEY = input('Pegá tu MISTRAL_API_KEY: ').strip()
    print('✅ API key configurada manualmente')

✅ API key cargada desde Colab Secrets


# CELDA 12 - Crear el AGENTE de LangChain (¡ESTA ES LA PARTE CLAVE!)






In [19]:
# Configurar el LLM
llm = ChatMistralAI(
    api_key=MISTRAL_API_KEY,
    model="mistral-small-latest",  # o "mistral-large-latest"
    temperature=0,  # 0 = respuestas consistentes, 0.7 = creativo
    max_retries=2
)

# Definir el prompt personalizado (prefix)
prefix = f"""
Eres un analista de datos experto en ventas. Tenés acceso a un DataFrame llamado 'df'
que contiene información de ventas de una empresa.

📊 INFORMACIÓN DEL DATASET:
- Filas: {df.shape[0]:,}
- Columnas: {', '.join(df.columns.tolist())}
- Período: {df['orderdate'].min().year if 'orderdate' in df.columns else 'N/A'} a {df['orderdate'].max().year if 'orderdate' in df.columns else 'N/A'}
- Países: {df['country'].nunique() if 'country' in df.columns else 'N/A'}
- Clientes únicos: {df['customername'].nunique() if 'customername' in df.columns else 'N/A'}

🎯 TUS CAPACIDADES:
Podés escribir y ejecutar código Python para:
1. Filtrar datos: df[df['country'] == 'USA']
2. Agrupar: df.groupby('productline')['sales'].sum()
3. Calcular estadísticas: df['sales'].mean(), .sum(), .max()
4. Ordenar: df.sort_values('sales', ascending=False).head(10)
5. Análisis temporal: df[df['year_id'] == 2003].groupby('month_id')['sales'].sum()

📋 REGLAS IMPORTANTES:
1. SIEMPRE usá el DataFrame 'df' (ya está cargado)
2. Mostrá el código que usaste para llegar a tu respuesta
3. Si el código da error, intentá otra aproximación
4. Para respuestas numéricas, formateá con separadores de miles y 2 decimales
5. Si no hay datos suficientes, decilo claramente
6. Respondé en español, pero el código en inglés (pandas está en inglés)

⚠ RESTRICCIONES:
- No modifiques el DataFrame original (no hagas asignaciones permanentes)
- Si necesitás copias, usá df_copy = df.copy()
- Las operaciones deben ser seguras y eficientes
"""

# Crear el agente (¡AQUÍ ESTÁ LO QUE BUSCABAS!)
agent_executor = create_pandas_dataframe_agent(
    llm,                        # Modelo de lenguaje
    df,                         # Tu DataFrame limpio
    prefix=prefix,              # Instrucciones personalizadas
    verbose=True,               # Muestra el razonamiento del agente
    allow_dangerous_code=True,  # Necesario para ejecutar pandas
    handle_parsing_errors=True, # Maneja errores automáticamente
    agent_type="tool-calling",  # Tipo de agente recomendado
    max_iterations=5,           # Evita loops infinitos
    early_stopping_method="generate",  # Genera respuesta si hay error
    number_of_head_rows=10,     # Muestra primeras 10 filas al agente
    max_execution_time=30       # Timeout de 30 segundos
)

print('\n' + '='*60)
print('🎉 AGENTE LANGCHAIN CREADO EXITOSAMENTE')
print('='*60)
print(f'✅ LLM: Mistral Small')
print(f'✅ DataFrame: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'✅ Agent Type: tool-calling')
print(f'✅ Max iterations: 5')
print('='*60)


🎉 AGENTE LANGCHAIN CREADO EXITOSAMENTE
✅ LLM: Mistral Small
✅ DataFrame: 2,823 filas × 26 columnas
✅ Agent Type: tool-calling
✅ Max iterations: 5


# CELDA 13 - Función de consulta simplificada

In [20]:
def consultar(pregunta, verbose=True):
    """
    Consulta al agente de LangChain

    Args:
        pregunta: str - Pregunta en lenguaje natural
        verbose: bool - Muestra el proceso del agente

    Returns:
        str - Respuesta del agente
    """
    try:
        resultado = agent_executor.invoke({"input": pregunta})
        return resultado['output']
    except Exception as e:
        return f"❌ Error al procesar la pregunta: {str(e)}"

def limpiar_historial():
    """Nota: LangChain no mantiene historial automáticamente,
       pero podemos recrear el agente si es necesario"""
    global agent_executor
    agent_executor = create_pandas_dataframe_agent(
        llm, df, prefix=prefix, verbose=True,
        allow_dangerous_code=True, handle_parsing_errors=True,
        agent_type="tool-calling", max_iterations=5
    )
    print("🔄 Agente reiniciado (historial limpiado)")

print('✅ Función consultar() lista')

✅ Función consultar() lista


# CELDA 14 - Bot interactivo

In [21]:
print('\n' + '='*55)
print('   🤖 SALES BOT - LANGCHAIN AGENT + MISTRAL')
print('='*55)
print('  Preguntame cualquier cosa sobre tus datos de ventas')
print('  Ejemplos:')
print('    • ¿Cuál es el país con más ventas?')
print('    • Top 5 clientes por revenue')
print('    • Ventas por línea de producto en 2003')
print('    • ¿Cuál es el producto más vendido?')
print('')
print('  Comandos: salir | reset | info')
print('='*55)
print()

while True:
    try:
        entrada = input('👤 Tú: ').strip()
    except (KeyboardInterrupt, EOFError):
        print('\n👋 ¡Hasta luego!')
        break

    if not entrada:
        continue

    # Comandos especiales
    if entrada.lower() == 'salir':
        print('👋 ¡Hasta luego!')
        break

    elif entrada.lower() == 'reset':
        limpiar_historial()
        continue

    elif entrada.lower() == 'info':
        print(f'\n📊 Información del dataset:')
        print(f'   • Filas: {df.shape[0]:,}')
        print(f'   • Columnas: {df.shape[1]}')
        print(f'   • Columnas: {", ".join(df.columns[:10])}...')
        print(f'   • Países: {df["country"].nunique() if "country" in df.columns else "N/A"}')
        print(f'   • Ventas totales: ${df["sales"].sum():,.2f}' if 'sales' in df.columns else '')
        print()
        continue

    # Consulta normal
    print('  ⏳ El agente está analizando...\n')
    respuesta = consultar(entrada, verbose=True)
    print(f'\n🤖 Bot:\n{respuesta}\n')


   🤖 SALES BOT - LANGCHAIN AGENT + MISTRAL
  Preguntame cualquier cosa sobre tus datos de ventas
  Ejemplos:
    • ¿Cuál es el país con más ventas?
    • Top 5 clientes por revenue
    • Ventas por línea de producto en 2003
    • ¿Cuál es el producto más vendido?

  Comandos: salir | reset | info

👤 Tú: ¿Cuál es el país con más ventas?
  ⏳ El agente está analizando...



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.groupby('country')['sales'].sum().sort_values(ascending=False).head(1)"}`


country
USA    3627982.83
Name: sales, dtype: float64El país con más ventas es **USA** con ventas totales de **3.627.982,83**.

> Finished chain.

🤖 Bot:
El país con más ventas es **USA** con ventas totales de **3.627.982,83**.

👤 Tú: salir
👋 ¡Hasta luego!


# CELDA 15 - Consulta individual rápida

In [22]:
# Cambiá la pregunta y ejecutá esta celda
pregunta = "¿Cuáles son los 3 países con mayores ventas y cuánto representa cada uno?"

respuesta = consultar(pregunta)
print(f"📊 Pregunta: {pregunta}\n")
print(f"🤖 Respuesta:\n{respuesta}")

# ================================================================
# CELDA 16 - Consultas en lote
# ================================================================
preguntas_lote = [
    "¿Cuál es el total de ventas por año?",
    "Top 5 clientes por volumen de compras",
    "¿Qué línea de producto genera más ingresos?",
    "Promedio de profit_margin por país",
    "¿Cuál es el mes con más ventas históricamente?"
]

print('\n' + '='*55)
print('  CONSULTAS EN LOTE')
print('='*55)

for i, pregunta in enumerate(preguntas_lote, 1):
    print(f'\n[{i}/{len(preguntas_lote)}] {pregunta}')
    print('-' * 50)
    respuesta = consultar(pregunta, verbose=False)
    print(f'✅ {respuesta}\n')

print('🎉 Lote de consultas completado')



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Agrupar por país y sumar las ventas, luego ordenar de mayor a menor y seleccionar los top 3\nventas_por_pais = df.groupby('country')['sales'].sum().sort_values(ascending=False).head(3)\nventas_por_pais"}`


country
USA       3627982.83
Spain     1215686.92
France    1110916.52
Name: sales, dtype: float64Los **3 países con mayores ventas** son:

1. **USA**: $3.627.982,83
2. **Spain**: $1.215.686,92
3. **France**: $1.110.916,52

---
**Código utilizado:**
```python
# Agrupar por país y sumar las ventas, luego ordenar de mayor a menor y seleccionar los top 3
ventas_por_pais = df.groupby('country')['sales'].sum().sort_values(ascending=False).head(3)
ventas_por_pais
```

> Finished chain.
📊 Pregunta: ¿Cuáles son los 3 países con mayores ventas y cuánto representa cada uno?

🤖 Respuesta:
Los **3 países con mayores ventas** son:

1. **USA**: $3.627.982,83
2. **Spain**: $1.215.686,92
3. **France**: $1.110.91